# fundeval 示例：从原始净值到评价结论

本 notebook 按正文（[docs/投资组合绩效评估.md](../docs/投资组合绩效评估.md)）的顺序，用第九部分“从原始数据到绩效结果”的 12 个月演示数据，
一步步走完：读取与清洗 → 数据质量 → 单期收益 → 收益与风险指标 → 尾部风险 → Alpha 回归 → Brinson 单期例题 → 持续监控 → `evaluate` 报告与三段结论。

- 前面各节**不联网**：演示收益在 `tests/data/worked_example.csv`，先还原出净值序列，再从净值重新算起。
- 联网示例集中在最后一节，由 `RUN_ONLINE = False` 控制，默认不执行。
- 不需要 matplotlib 或 Jupyter 以外的依赖；`pytest tests/test_notebook.py` 会在临时目录中按顺序执行全部代码单元，确保本 notebook 始终能跑通。

数据为教学演示，不代表真实基金业绩。

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

RUN_ONLINE = False  # 改为 True 才执行最后一节的联网示例（需要 pip install "fundeval[data]"）

# 找到演示数据：环境变量 FUNDEVAL_WORKED_EXAMPLE，或从当前目录向上找 tests/data/worked_example.csv
_found = os.environ.get("FUNDEVAL_WORKED_EXAMPLE")
if _found is None:
    _found = next(
        (str(p / "tests" / "data" / "worked_example.csv") for p in [Path.cwd(), *Path.cwd().parents]
         if (p / "tests" / "data" / "worked_example.csv").exists()),
        None,
    )
if _found is None:
    raise FileNotFoundError("找不到 tests/data/worked_example.csv：请在仓库目录内运行，或设置环境变量 FUNDEVAL_WORKED_EXAMPLE")
DATA = Path(_found)
ROOT = DATA.parents[2]

try:
    import fundeval
except ImportError:  # 未安装时直接使用仓库源码
    sys.path.insert(0, str(ROOT / "src"))
    import fundeval

print("fundeval", fundeval.__version__, "；演示数据：", DATA.name)

## 1 读取与清洗（正文第一部分第 2 节）

正文的清洗顺序为：日期对齐 → 币种统一 → 现金流识别 → 缺失与异常复核 → 收益计算。

演示数据是正文第九部分表格中的月度收益（组合、基准、无风险收益，百分数）。为了演示“从原始净值开始”，
先把组合与基准收益还原成以 1.0000 起步的净值（期初日取第一个月之前的月末），之后只用净值。

In [ ]:
from fundeval.etl import clean, schema
from fundeval.etl.sources import files

COLUMNS = {"月份": "date", "组合收益": "portfolio", "基准收益": "benchmark", "无风险收益": "risk_free"}
demo = files.load_returns(DATA, columns=COLUMNS)  # 百分数自动换算为小数

start = demo.index[0] - pd.offsets.MonthEnd(1)
nav = pd.concat([pd.Series({start: 1.0}), (1 + demo["portfolio"]).cumprod()]).rename("nav")
bench_level = pd.concat([pd.Series({start: 1.0}), (1 + demo["benchmark"]).cumprod()]).rename("benchmark")
risk_free = demo["risk_free"]

# 清洗：去重、对齐，校验净值（日期唯一升序、严格为正、无缺失）
nav = schema.validate_nav(clean.drop_duplicate_dates(nav))
levels = clean.align(nav, bench_level, names=["nav", "benchmark"])
print(levels.round(4).to_string())

现金流识别：本例是净值序列，不涉及申购赎回。若数据是账户资产或基金份额，可用 `clean.infer_cashflows`
按“份额变动 × 单位净值”推算净申赎，再用 `clean.flag_large_cashflows` 标出超过资产规模 5% 的大额现金流——
正文要求期中大额现金流按发生时点拆分子期。下面用一组虚构的份额演示。

In [ ]:
units = pd.Series([100.0, 100.0, 130.0, 128.0], index=nav.index[:4])  # 虚构份额：第 2 个月净申购 30 份
flows = clean.infer_cashflows(units, nav.iloc[:4])
assets = (units * nav.iloc[:4]).shift(1).reindex(flows.index)  # 期初资产 = 上期末份额 × 上期末净值
print(clean.flag_large_cashflows(flows, assets).round(4).to_string())

## 2 数据质量（正文第一部分）

`data_quality_report` 汇总缺失、异常收益（稳健 z 值）、疑似停牌或估值滞后与样本长度。缺失值只标记、不填零。
12 个月的演示样本少于 36 期，会被标为“样本较短，统计推断与能力判断受限”。

In [ ]:
from fundeval.etl import quality

nav_returns = (nav / nav.shift(1) - 1).dropna()
qrep = quality.data_quality_report(
    pd.DataFrame({"portfolio": nav_returns, "benchmark": demo["benchmark"]}), nav=nav, periods_per_year=12
)
print(qrep.summary().to_string(index=False))
print()
for issue in qrep.issues():
    print("-", issue)

## 3 单期收益（正文第二部分第 1 节）

复权净值的单期收益 r_t = NAV_t / NAV_{t−1} − 1。由净值重新算出的收益应与正文表格一致。

In [ ]:
from fundeval.etl.returns import nav_to_returns, price_to_returns

p = nav_to_returns(nav)
b = price_to_returns(bench_level).rename("benchmark")
assert np.allclose(p.to_numpy(), demo["portfolio"].to_numpy())
assert np.allclose(b.to_numpy(), demo["benchmark"].to_numpy())
print(pd.DataFrame({"组合": p, "基准": b, "无风险": risk_free}).map("{:.2%}".format).to_string())

## 4 收益与风险指标（正文第二、三部分）

- 累计收益几何连乘；年化收益几何口径
- 波动率、Sharpe、TE、IR 用同频算术均值与样本标准差（n−1），乘以 √K 年化（K = 12）
- 最大回撤的峰值含期初；M² 按基准波动率调整，M² 差额减基准年化算术均值

核对值（正文第九部分）：累计收益 10.2058%、Sharpe 1.1254、IR 2.2327、最大回撤 3.00%、M² 8.4668%。

In [ ]:
from fundeval import returns as ret
from fundeval import risk

K = 12
metrics = {
    "累计收益（组合）": ret.cumulative_return(p),
    "累计收益（基准）": ret.cumulative_return(b),
    "几何年化收益": ret.annualized_return(p, K),
    "年化波动率": risk.volatility(p, K),
    "最大回撤": risk.max_drawdown(p),
    "跟踪误差 TE": risk.tracking_error(p, b, K),
    "M²": risk.m_squared(p, b, risk_free, K),
}
ratios = {
    "Sharpe": risk.sharpe_ratio(p, risk_free, K),
    "信息比率 IR": risk.information_ratio(p, b, K),
}
for name, value in metrics.items():
    print(f"{name:<12}{value:>10.4%}")
for name, value in ratios.items():
    print(f"{name:<12}{value:>10.4f}")

assert round(metrics["累计收益（组合）"], 6) == 0.102058
assert round(ratios["Sharpe"], 4) == 1.1254 and round(ratios["信息比率 IR"], 4) == 2.2327
assert round(metrics["最大回撤"], 4) == 0.0300 and round(metrics["M²"], 6) == 0.084668

## 5 尾部风险（正文第八部分）

Sortino 的最低可接受收益 MAR 须事前约定，这里取每月无风险收益 0.15%；VaR 与 ES 为历史模拟，损失以正数报告。
12 个观测的 95% 尾部只有 0.6 个观测，结果只用于演示计算。

In [ ]:
from fundeval import tail

print(f"Sortino（MAR = 每月 0.15%）：{tail.sortino_ratio(p, 0.0015, K):.4f}")
print(f"Calmar：{tail.calmar_ratio(p, K):.4f}")
print(f"历史 VaR（95%）：{tail.historical_var(p, 0.95):.2%}")
print(f"历史 ES（95%）：{tail.historical_es(p, 0.95):.2%}")

## 6 Alpha 回归（正文第四部分第 2 节）

CAPM：r_p − r_f = α + β(r_m − r_f) + ε。演示数据没有单独的市场收益，用基准代替市场。
12 个样本期的 t 值只说明“观察到模型未解释的收益”，不足以判断管理能力。

In [ ]:
from fundeval.alpha.regression import capm_regression

capm = capm_regression(p, b, risk_free)
print(f"Alpha（每月）{capm.alpha:.4%}，算术年化 {capm.annualized_alpha(K):.2%}，t = {capm.alpha_t:.2f}，p = {capm.alpha_p:.3f}")
print(f"Beta {capm.betas['market']:.3f}，R² {capm.rsquared:.3f}，n = {capm.n}")

## 7 Brinson 单期例题（正文第五部分第 2 节）

正文的三行业例题：组合收益 2.00%、基准收益 1.50%，主动收益 0.50 个百分点拆为配置 0.20、选择 0.22、交互 0.08 个百分点（BHB）。
真实持仓可用 `schema.validate_holdings` 校验、`schema.sector_weights` 汇总为行业权重后直接输入。

In [ ]:
from fundeval.attribution import brinson_single

sectors = ["甲", "乙", "丙"]
wp = pd.Series([0.50, 0.30, 0.20], index=sectors)
wb = pd.Series([0.40, 0.40, 0.20], index=sectors)
rp = pd.Series([0.038, 0.010, -0.010], index=sectors)
rb = pd.Series([0.030, 0.010, -0.005], index=sectors)
bhb = brinson_single(wp, wb, rp, rb)
print((bhb.table() * 100).round(2).replace(-0.0, 0.0).to_string())  # 单位：百分点
assert round(bhb.totals["allocation"] * 100, 2) == 0.20 and round(bhb.totals["selection"] * 100, 2) == 0.22

## 8 持续监控（正文第七部分）

正文演示：目标年化主动收益 3%、目标年化 TE 4%，K = 252、h = 21 个交易日；月内算术主动收益合计 −2.00% 时，
z = (−2.00% − 0.25%) / 1.155% ≈ −1.95，风险倍数正常时仍为 Green。阈值为正文演示值，须按策略校准。

In [ ]:
from fundeval import monitor

h = 21
z = monitor.z_score(-0.02, h, 0.03, 0.04, 252)
print(f"期望主动收益 {monitor.expected_active_return(0.03, h, 252):.2%}，目标风险 {monitor.target_risk(0.04, h, 252):.3%}，z = {z:.2f}")
print("分区：", monitor.classify(1.0, z))

## 9 evaluate 报告与三段结论（正文第九、十部分）

`evaluate` 把上面各步汇总成一份报告：口径、六个维度的指标、回归、数据质量与三段结论。
评价口径可以集中写在 `EvaluationConfig`（或 TOML 文件，见 `examples/config.toml`）里，关键字参数与 config 同时给出时关键字参数优先。

In [ ]:
from fundeval.config import EvaluationConfig
from fundeval.report import evaluate, to_markdown

config = EvaluationConfig(freq="M", fees="费用后净值（演示）", confidence=0.95)
report = evaluate(
    pd.DataFrame({"portfolio": p, "benchmark": b, "risk_free": risk_free}),
    labels={"title": "正文第九部分演示组合", "portfolio": "演示组合", "benchmark": "演示基准", "risk_free": "每月 0.15%"},
    config=config,
)
print(report.section("风险效率")[["指标", "数值"]].to_string(index=False))
print()
print(report.conclusion())

In [ ]:
# 完整 Markdown 报告（写到文件可用 to_markdown(report, "report.md")）
markdown = to_markdown(report)
print(markdown[:1500])

## 10 联网示例（默认不执行）

把第一个代码单元中的 `RUN_ONLINE` 改为 `True` 后执行。需要 `pip install "fundeval[data]"`，并能访问东方财富、中证指数官网与
Kenneth R. French Data Library：

- 只给基金代码：取净值与分红、合同业绩比较基准与无风险利率，生成同样的报告
- A 股四因子 `cn_index_proxy4`（UMD = 沪深300动量 H30260 − 沪深300 000300，均为价格指数）
- 美国市场因子 `carhart_us`（French 因子库，美元计价，适合投资美股的 QDII）

In [ ]:
if RUN_ONLINE:
    from fundeval.etl.sources import french
    from fundeval.report.inputs import ReportOptions, build_report_inputs

    online = evaluate(**build_report_inputs(ReportOptions(
        fund="110020", start="2021-01-01", end="2025-12-31", factors="cn_index_proxy4", hac_lags=3,
    )))
    print(online.conclusion())

    carhart = french.carhart_factors()
    print(carhart.tail())
else:
    print("RUN_ONLINE = False：跳过联网示例")